# A.8 同一條線索，放在哪裡會有差嗎？


把下面八筆活動室紀錄交給助理，問「攝影社的活動室代碼是什麼？只回答代碼」。紙上答案是 `M17`：代碼是房間的標籤，不必猜它代表哪一層樓。現在保留同一份材料，把提供答案的那筆紀錄移到資料的開頭、中間或結尾，三次都應回答 `M17`。這樣可以問一件比「輸入放得下」更進一步的事：位置改了，助理還能穩定用到那條線索嗎？

| 紀錄標籤 | 原文 |
| --- | --- |
| T | 攝影社的活動室代碼是 M17。 |
| U1 | 棋藝社的活動室代碼是 B04。 |
| U2 | 合唱社的活動室代碼是 C26。 |
| U3 | M17 在二樓。 |
| U4 | B04 在一樓。 |
| U5 | C26 在三樓。 |
| U6 | 美術社的活動室代碼是 E31。 |
| U7 | E31 在二樓。 |

T、U1 這些標籤讓我們能追蹤紀錄，並不是活動室代碼。這一題只需要 T；其他紀錄雖與活動室有關，都不能替代攝影社與 `M17` 的對應。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rewrite-A-clue-position.svg")))

圖中每格是一筆紀錄，不是一個 token。先將 T 放在第一筆；再拿起同一張 T，插到第四筆；最後移到第八筆。U1 到 U7 的相對順序保持不變，沒有新增公告、改寫線索或另換問題。這裡的「首、中、尾」指**資料段**的位置，問題始終放在資料段之後。因此尾端版本的線索比較靠近問題，這正是要觀察的位置差異，不是把問題也一起搬動。

若真的拿模型比較，三版要使用相同的完整輸入長度、同一句問題、同一句線索，以及相同的新回答上限。依 [A.7](https://birdhackor.github.io/tiny-perceptron-vlm/A.7.html) 的方法，先完成對話格式的組合，再核對三版的 token 數一致，且八筆紀錄都完整留下；中文字數相同還不夠。模型權重、解碼方法與其他設定也保持固定。若重排後 token 數不同，先調整三版共用的紀錄分隔格式，重新核對長度，再開始比較，不能把多出的長度當成位置效應。

紙上可以先畫三欄「首、中、尾」，每欄填上期待答案 `M17`。之後若有實際生成，再各自記下原始回答、代碼是否正確，以及回答是否完整結束。例如 `M17` 是正確內容；`B04` 是取錯紀錄；只輸出 `M` 而耗盡回答上限，還有未完成輸出的問題。把這些差別留下來，比只記「成功／失敗」更容易找出要檢查的地方。

還可以另外準備只含 T 與同一問題的短版。短版若也答錯，就先檢查模型是否懂這個提問及代碼格式；短版答對而長版失敗，才有理由繼續查長度、位置或幹擾。短版是另一個診斷條件，不與三個等長版本混算。

在多題比較中，若中間位置常比首尾位置差，可以描述「這個模型在這組題目上對位置敏感」。也可能三處都答對，或出現別的排列。位置測試的用途是把差異量出來，沒有預先指定每個模型都必須畫出中間較低的 U 形曲線。即使三處都成功，也只驗收了這種單線索取回；下一節會問，需要多筆資訊時又該怎麼查。

<details>
<summary>補充：來源與例子的界線</summary>

本節的位置控制參考 Liu et al., [*Lost in the Middle: How Language Models Use Long Contexts*，arXiv:2307.03172v3](https://arxiv.org/pdf/2307.03172v3)，§2.1–2.3、§3.1。原論文在多個當時模型與設定觀察到首尾較好、中間較差，也有部分合成取回任務接近全對。本節的八筆紀錄是教學用紙上材料，沒有重現原論文評測，也沒有提供本課模型的位置成績。

</details>
